In [ ]:
import re
import math
import pandas as pd
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor
!pip install unbabel-comet
from comet import download_model, load_from_checkpoint

Что в нашем кейсе важнее всего?
Для успешного решения задачи нужно чтобы итоговый файл

1.   Запустился: не побитая разметка
2.   Отрендерился: не побитые теги
3.   Сохранил исходную информацию: не потерял теги и слова
4.   Был приемлемым для игры: имел достаточное кач-во перевода

Идея метрики такова: нужно посмотреть на качество перевода и качество сохранения тегов.
Для этого построим дерево тегов (чтобы оценить сохранение структуры) и посмотрим на общее сохранение тегов.

In [ ]:
# $VAR$, $VALUE|Y$, $NUM|2%$, ...
DOLLAR_VAR_RE = re.compile(r"\$[A-Za-z0-9_\|\.\+\-\%]+\$")
# £icon£
POUND_ICON_RE = re.compile(r"£[^£\s]+£")
# [Root.GetAdj], [This.GetName], [Root.Owner.GetSpecies.GetName], ...
SCRIPT_TAG_RE = re.compile(r"\[[^\]]+\]")
# §Y, §G, ...
COLOR_OPEN_RE = re.compile(r"§[A-Z]")
# §!
COLOR_CLOSE_RE = re.compile(r"§!")


def strip_tags_for_translation(text: str) -> str:
    """
    Удаление тегов перед оценкой качества перевода:
    - цветовые теги §X и §! убираются, но их содержимое остаётся;
    - плейсхолдеры/иконки/скрипты ($VAR$, £icon£, [Root.GetAdj]) выкидываются целиком.
    """
    t = DOLLAR_VAR_RE.sub(" ", text)
    t = POUND_ICON_RE.sub(" ", t)
    t = SCRIPT_TAG_RE.sub(" ", t)

    t = COLOR_OPEN_RE.sub("", t)
    t = COLOR_CLOSE_RE.sub("", t)

    t = re.sub(r"\s+", " ", t).strip()
    return t

@dataclass
class ColorSpan:
    start: int  # позиция по символам (начало маркера открытия)
    end: int    # позиция по символам (начало маркера закрытия)
    code: str   # буква цвета, например 'Y'


@dataclass
class TagToken:
    start: int
    end: int
    kind: str         # 'color_open', 'color_close', 'dollar', 'pound', 'script'
    payload: str = "" # код цвета / имя переменной / имя иконки / тело скрипта


@dataclass
class TagNode:
    kind: str
    value: Optional[str] = None
    children: List["TagNode"] = field(default_factory=list)


@dataclass
class TagSummary:
    root: TagNode

    n_color_spans: int
    n_color_open: int
    n_color_close: int
    n_dollar_vars: int
    n_pound_icons: int
    n_script_tags: int
    color_spans: List[ColorSpan]

    malformed_colors: bool
    malformed_placeholders: bool
    malformed_icons: bool
    malformed_scripts: bool

    @property
    def malformed_any(self) -> bool:
        """
        Есть ли какие-либо явные ошибки разметки,
        из-за которых файл может не запуститься.
        """
        return (
            self.malformed_colors
            or self.malformed_placeholders
            or self.malformed_icons
            or self.malformed_scripts
        )


Вкладывать можно только в теги цветов. Проверим и корректность всей структуры.

In [ ]:

def detect_illegal_nesting(text: str) -> Tuple[bool, bool, bool]:
    """
    Проверяет, есть ли внутри атомарных тегов ($...$, £...£, [....])
    какие-либо другие теги.
    Разрешённые вложения:
      - любые теги внутри цветового тега §X ... §!
    Запрещенные:
      - любой тег внутри $...$
      - любой тег внутри £...£
      - любой тег внутри [....]
    """
    tokens = extract_tag_tokens(text)

    malformed_placeholders = False
    malformed_icons = False
    malformed_scripts = False

    for i, outer in enumerate(tokens):
        if outer.kind not in ("dollar", "pound", "script"):
            continue

        for j, inner in enumerate(tokens):
            if i == j:
                continue

            if outer.start <= inner.start and inner.end <= outer.end:
                if outer.kind == "dollar":
                    malformed_placeholders = True
                elif outer.kind == "pound":
                    malformed_icons = True
                elif outer.kind == "script":
                    malformed_scripts = True

    return malformed_placeholders, malformed_icons, malformed_scripts

def summarize_tags(text: str) -> TagSummary:
    """
    Строит сводку по тегам в строке и проверяет побитость разметки.
    Здесь же строится дерево тегов
    """
    color_spans, malformed_colors_spans = find_color_spans(text)

    root, malformed_colors_tree = build_tag_tree(text)
    malformed_colors = malformed_colors_spans or malformed_colors_tree

    n_dollar = len(DOLLAR_VAR_RE.findall(text))
    n_pound  = len(POUND_ICON_RE.findall(text))
    n_script = len(SCRIPT_TAG_RE.findall(text))
    n_open   = len(COLOR_OPEN_RE.findall(text))
    n_close  = len(COLOR_CLOSE_RE.findall(text))

    # $ ... $ : каждая переменная даёт ровно два символа '$'
    n_dollar_chars = text.count("$")
    malformed_placeholders = (n_dollar_chars != 2 * n_dollar)

    # £icon£ : каждый маркер даёт два символа '£'
    n_pound_chars = text.count("£")
    malformed_icons = (n_pound_chars != 2 * n_pound)

    # [script] : каждый скрипт предполагает одну '[' и одну ']'
    n_lbrackets = text.count("[")
    n_rbrackets = text.count("]")
    malformed_scripts = (n_lbrackets != n_script) or (n_rbrackets != n_script)

    il_malformed_placeholders, il_malformed_icons, il_malformed_scripts = detect_illegal_nesting(text)

    malformed_placeholders = malformed_placeholders or il_malformed_placeholders
    malformed_icons        = malformed_icons        or il_malformed_icons
    malformed_scripts      = malformed_scripts      or il_malformed_scripts

    return TagSummary(
        root=root,
        n_color_spans=len(color_spans),
        n_color_open=n_open,
        n_color_close=n_close,
        n_dollar_vars=n_dollar,
        n_pound_icons=n_pound,
        n_script_tags=n_script,
        color_spans=color_spans,
        malformed_colors=malformed_colors,
        malformed_placeholders=malformed_placeholders,
        malformed_icons=malformed_icons,
        malformed_scripts=malformed_scripts,
    )


def find_color_spans(text: str) -> Tuple[List[ColorSpan], bool]:
    """
    Находит пары §X ... §! и возвращает список интервалов color spans.
    Если структура цветовых тегов нарушена — malformed=True.
    """
    events = []
    for m in COLOR_OPEN_RE.finditer(text):
        # ('open', position, 'Y')
        events.append(("open", m.start(), m.group()[1]))
    for m in COLOR_CLOSE_RE.finditer(text):
        # ('close', position, None)
        events.append(("close", m.start(), None))

    events.sort(key=lambda x: x[1])
    stack: List[Tuple[str, int]] = []
    spans: List[ColorSpan] = []
    malformed = False

    for kind, pos, code in events:
        if kind == "open":
            stack.append((code, pos))
        else:
            if not stack:
                malformed = True
                continue
            open_code, open_pos = stack.pop()
            span = ColorSpan(start=open_pos, end=pos, code=open_code)
            spans.append(span)

    if stack:
        # остались незакрытые открывающие
        malformed = True

    return spans, malformed


def extract_tag_tokens(text: str) -> List[TagToken]:
    """
    Извлекает все теги как плоский список токенов с позициями.
    """
    tokens: List[TagToken] = []

    for m in COLOR_OPEN_RE.finditer(text):
        tokens.append(TagToken(start=m.start(), end=m.end(),
                               kind="color_open", payload=m.group(0)[1]))
    for m in COLOR_CLOSE_RE.finditer(text):
        tokens.append(TagToken(start=m.start(), end=m.end(),
                               kind="color_close", payload=""))

    for m in DOLLAR_VAR_RE.finditer(text):

        payload = m.group(0)[1:-1]
        tokens.append(TagToken(start=m.start(), end=m.end(),
                               kind="dollar", payload=payload))

    for m in POUND_ICON_RE.finditer(text):

        payload = m.group(0)[1:-1]
        tokens.append(TagToken(start=m.start(), end=m.end(),
                               kind="pound", payload=payload))

    for m in SCRIPT_TAG_RE.finditer(text):

        payload = m.group(0)[1:-1]
        tokens.append(TagToken(start=m.start(), end=m.end(),
                               kind="script", payload=payload))

    tokens.sort(key=lambda t: t.start)
    return tokens


def build_tag_tree(text: str) -> Tuple[TagNode, bool]:
    """
    Строит дерево тегов:
    - цвета образуют иерархию по §X ... §!;
    - $VAR$, £icon£, [Root.Get...] становятся листьями у текущего цвета/корня.

    Возвращает (root, malformed_colors).
    malformed_colors = True, если структура цветовых тегов сломана
    """
    tokens = extract_tag_tokens(text)

    root = TagNode(kind="root")
    stack: List[TagNode] = [root]
    malformed = False

    for tok in tokens:
        parent = stack[-1]

        if tok.kind == "color_open":
            node = TagNode(kind="color", value=tok.payload)
            parent.children.append(node)
            stack.append(node)

        elif tok.kind == "color_close":
            # Закрываем последний открытый цвет
            if len(stack) == 1 or stack[-1].kind != "color":
                # Закрывающий цвет без открывающего или сломанный стек
                malformed = True
            else:
                stack.pop()

        elif tok.kind == "dollar":
            parent.children.append(TagNode(kind="dollar", value=tok.payload))

        elif tok.kind == "pound":
            parent.children.append(TagNode(kind="pound", value=tok.payload))

        elif tok.kind == "script":
            parent.children.append(TagNode(kind="script", value=tok.payload))

    # Если остались незакрытые цветовые теги
    if len(stack) > 1:
        malformed = True
        stack[:] = [root]

    return root, malformed

Блок оценивает корректность тегов в переводе и отвечает за три вещи:

1) **tag_type_recall** — считает полноту по каждому типу тегов.  
   Если в переводе отсутствуют теги, которые были в эталоне, это снижает recall.

2) **weighted_tag_content_score** — вычисляет итоговый скор сохранности тегов с разными весами:  цветовые теги важны меньше, иконки — больше, а переменные и скрипты — самые важные.  Итог — взвешенное среднее recall по типам тегов.

3) **tree_structure_score** — сравнивает структуру вложения тегов.  
   Строятся пути дерева от корня до каждого тега, и затем считается совпадение  
   путей между эталоном и кандидатом.

4) **compilability_score** — проверяет корректность разметки.  
   Если есть ошибки (незакрытые теги, запрещённые вложения), возвращает 0.


In [ ]:
def tag_type_recall(ref: TagSummary, cand: TagSummary) -> Dict[str, float]:
    """
    Recall по наличию тегов каждого типа.
    Штрафует за отсутствие тега в кандидате.
    """
    def safe_recall(gold: int, pred: int) -> float:
        if gold == 0:
            return 1.0
        return max(0.0, min(1.0, pred / gold))

    return {
        "color_spans":  safe_recall(ref.n_color_spans, cand.n_color_spans),
        "dollar_vars":  safe_recall(ref.n_dollar_vars, cand.n_dollar_vars),
        "pound_icons":  safe_recall(ref.n_pound_icons, cand.n_pound_icons),
        "script_tags":  safe_recall(ref.n_script_tags, cand.n_script_tags),
    }


def weighted_tag_content_score(ref: TagSummary, cand: TagSummary) -> float:
    """
    Оценивает сохранность тегов по типам, с разными весами:
    Выделение (цвет) < иконка < подстановка/скрипт.
    """
    recalls = tag_type_recall(ref, cand)

    weights = {
        "color_spans": 1.0,  # выделение
        "pound_icons": 2.0,  # иконки
        "dollar_vars": 3.0,  # подстановки
        "script_tags": 3.0,  # скрипты
    }

    num = 0.0
    den = 0.0
    for t, w in weights.items():
        num += w * recalls[t]
        den += w
    if den == 0:
        return 1.0
    return num / den


def collect_tag_paths(node: TagNode,
                      prefix: Tuple[str, ...],
                      paths: Counter) -> None:
    """
    Собирает пути вида ("color:Y", "dollar:STAR_NAME") для всех тегов.
    root сам по себе не учитываем как шаг пути.
    """
    if node.kind == "root":
        cur_prefix = prefix
    else:
        label = f"{node.kind}:{node.value}" if node.value is not None else node.kind
        cur_prefix = prefix + (label,)

    if not node.children:
        if cur_prefix:
            paths[cur_prefix] += 1
    else:
        for ch in node.children:
            collect_tag_paths(ch, cur_prefix, paths)


def tree_structure_score(ref_root: TagNode, cand_root: TagNode) -> float:
    """
    Сравниваем структуры деревьев тегов:
    строим мультисеты путей и считаем recall относительно рефа.
    """
    ref_paths: Counter = Counter()
    cand_paths: Counter = Counter()

    collect_tag_paths(ref_root, tuple(), ref_paths)
    collect_tag_paths(cand_root, tuple(), cand_paths)

    if not ref_paths:
        # В рефе тегов нет – структура не важна
        return 1.0

    intersect = 0
    total = sum(ref_paths.values())
    for p, cnt_ref in ref_paths.items():
        cnt_cand = cand_paths.get(p, 0)
        intersect += min(cnt_ref, cnt_cand)

    return max(0.0, min(1.0, intersect / total))


def tag_structure_score(ref: TagSummary, cand: TagSummary) -> float:
    """
    Структурное сходство дерева тегов:
    учитывает, какие теги вложены в какие цвета и в каком контексте находятся.
    """
    return tree_structure_score(ref.root, cand.root)


def compilability_score(tags: TagSummary) -> float:
    """
    1.0 если нет явных ошибок разметки (по типам тегов),
    0.0 если есть.
    """
    return 0.0 if tags.malformed_any else 1.0


def text_penalty(text_score: float, sharpness: float = 0.8, cutoff: float = 0.7) -> float:
    """
    Логарифмический штраф по качеству перевода, применяемый только
    до заданного порога cutoff. После cutoff штраф не используется.

    text_score ∈ [0,1]
    cutoff — порог, после которого penalty = 1.0
    """

    # защита от выхода за пределы
    t = max(0.0, min(1.0, text_score))

    # если перевели достаточно хорошо — штраф не применяется
    if t >= cutoff:
        return t

    # иначе логарифмический штраф
    s = max(1e-6, min(0.999999, sharpness))
    penalty = -math.log(1.0 - s * t) / -math.log(1.0 - s)

    return penalty

Для оценки качества перевода используется обучаемая нейронная метрика COMET (Crosslingual Optimized Metric for Evaluation of Translation). У COMET, неплохая корреляция с человеческими оценками.

В работе применяется библиотека unbabel-comet, поддерживаемая исследовательской группой Unbabel и доступная на GitHub и PyPI:
https://github.com/Unbabel/COMET

В качестве модели используется Unbabel/wmt22-comet-da. Она выдает скор в диапазоне от 0 до 1.



In [ ]:
#для чистого вывода
import os
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning").setLevel(logging.ERROR)
os.environ["COMET_DISABLE_PROGRESS_BAR"] = "1"

class CometScorer:
    """
    Ожидается модель 22 года или новее
    Возвращает скор в [0,1].
    """

    def __init__(self, comet_model):
        self.model = comet_model

    def score(self, src: Optional[str], ref: str, cand: str) -> float:
        """
        Семантический скор перевода в диапазоне [0,1].
        COMET использует исходное предложение, перевод и эталон.
        """
        src_text = strip_tags_for_translation(src) if src is not None else ""
        ref_text = strip_tags_for_translation(ref)
        cand_text = strip_tags_for_translation(cand)

        data = [{
            "src": src_text,
            "mt": cand_text,
            "ref": ref_text,
        }]

        out = self.model.predict(data, batch_size=8, gpus=0, progress_bar=False)

        score = float(out["scores"][0])


        return score

Функция **evaluate_translation** объединяет результаты всех подметрик и рассчитывает итоговый показатель качества перевода.
Оценка строится на четырёх компонентах:

1) Компилируемость (comp_score) — проверяет, нет ли ошибок во вложении тегов и может ли строка корректно загрузиться в игре.
   Если строка некорректна, итоговый балл принудительно обнуляется.

2) Сохранность тегов (tag_score) — комбинирует две подметрики:
*   tag_content_score — насколько полно в кандидате сохранены все типы тегов
*   tag_structure_score — насколько совпадает древовидная структура и вложенность тегов.
Сохранность тегов для нас важнее структуры.

3) Качество перевода (text_score) — семантическая метрика от COMET.
   На её основе вычисляется логарифмический штраф text_penalty(text_score), чтобы слабый перевод сильнее понижал итоговый балл.

4) Финальный результат — перемножение всех трёх факторов:
      `final = compilability * tag_score * text_penalty(text_score)`


In [ ]:
def evaluate_translation(
    ref: str,
    cand: str,
    comet_scorer: CometScorer,
    src: Optional[str] = None,
) -> Dict[str, float]:
    """
    Главная функция.

    ref  — эталонный перевод.
    cand — перевод модели.
    src  — исходная строка (на языке источника), для COMET.
    comet_scorer — экземпляр CometScorer; если None — text_score=0.0.
    """
    # Качество перевода
    text_score = comet_scorer.score(src, ref, cand)

    # Теги
    ref_tags = summarize_tags(ref)
    cand_tags = summarize_tags(cand)

    tag_content = weighted_tag_content_score(ref_tags, cand_tags)
    tag_struct = tag_structure_score(ref_tags, cand_tags)

    # Контент тегов важнее структуры, но оба учитываем
    tag_score = 0.6 * tag_content + 0.4 * tag_struct

    # Компилируемость (запустится / не запустится)
    comp_score = compilability_score(cand_tags)

    # Финальный скор
    final = comp_score * tag_score * text_penalty(text_score)

    return {
        "text_score": text_score,
        "tag_content_score": tag_content,
        "tag_structure_score": tag_struct,
        "tag_score": tag_score,
        "compilability": comp_score,
        "final_score": final,
    }

In [ ]:
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)
scorer = CometScorer(comet_model)

ref = 'Мы открыли §Y$STAR_NAME$§! в системе £star£ — [Root.GetAdj] звезда.'
good = 'Мы обнаружили §Y$STAR_NAME$§! в системе £star£ — [Root.GetAdj] звезда.'
missing_var = 'Мы обнаружили §Yзвезду§! в системе £star£ — [Root.GetAdj] $STAR_NAME$.'
broken_color = 'Мы обнаружили §Y$STAR_NAME$ в системе £star£ — [Root.GetAdj] звезда.'  # нет §!

src = "We have discovered a new star called $STAR_NAME$ in the system."

results = {
    "good": evaluate_translation(ref, good, src=src, comet_scorer=scorer),
    "missing_var": evaluate_translation(ref, missing_var, src=src, comet_scorer=scorer),
    "broken_color": evaluate_translation(ref, broken_color, src=src, comet_scorer=scorer),
}

df = pd.DataFrame(results).T
df = df[["text_score",
         "tag_content_score",
         "tag_structure_score",
         "tag_score",
         "compilability",
         "final_score"]]

display(df.round(3))

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


,text_score,tag_content_score,tag_structure_score,tag_score,compilability,final_score
good,0.883,1.000,1.000,1.000,1.0,0.883
missing_var,0.831,1.000,0.667,0.867,1.0,0.720
broken_color,0.883,0.889,0.333,0.667,0.0,0.000


Сейчас подготовим функцию для оценки набора значений, а не одной строки. Чтобы сожно было следить за прогрессом и процесс шел быстрее.
И чтобы память коллаба не забивалась держать батчи будем в импровизированном кеше (тогда можно будет и при рестарте не терять прогресс)

In [ ]:
import hashlib
from pathlib import Path
import threading

def triple_hash(src: Optional[str], ref: str, cand: str) -> str:
    """
    Считаем хеш по трём строкам
    """
    h = hashlib.sha256()
    src = src or ""
    for part in (src, ref, cand):
        h.update(part.encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()

@dataclass
class CometFileCache:
    """
    Кеш COMET-скоров в текстовом файле.
    Формат строки: "<hash>\\t<score>\\n"
    """
    path: Path
    use_existing: bool = True         # брать старые значения из файла при старте
    flush_every: int = 1              # через сколько записей делать flush (1 = после каждой)

    _data: Dict[str, float] = field(default_factory=dict, init=False)
    _buffer: list = field(default_factory=list, init=False)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        self.path = Path(self.path)
        if self.use_existing and self.path.exists():
            with self.path.open("r", encoding="utf-8") as f:
                for line in f:
                    line = line.rstrip("\n")
                    if not line:
                        continue
                    h, s = line.split("\t")
                    self._data[h] = float(s)

    def get(self, h: str) -> Optional[float]:
        return self._data.get(h)

    def has(self, h: str) -> bool:
        return h in self._data

    def add(self, h: str, score: float):
        with self._lock:
            self._data[h] = score
            self._buffer.append((h, score))
            if len(self._buffer) >= self.flush_every:
                self._flush_locked()

    def _flush_locked(self):
        if not self._buffer:
            return
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self.path.open("a", encoding="utf-8") as f:
            for h, s in self._buffer:
                f.write(f"{h}\t{s}\n")
        self._buffer.clear()

    def flush(self):
        with self._lock:
            self._flush_locked()


class ModelCometScorer:
    """
    Обёртка над COMET-моделью с поддержкой кеша.
    """

    def __init__(self, comet_model, default_batch_size: int = 32,
                 cache: Optional[CometFileCache] = None,
                 reuse_old_cache: bool = True):
        self.model = comet_model
        self.default_batch_size = default_batch_size
        self.cache = cache
        self.reuse_old_cache = reuse_old_cache  # bool: брать ли значения из старого файла

    def score_batch(
        self,
        src_list: List[Optional[str]],
        ref_list: List[str],
        cand_list: List[str],
        batch_size: Optional[int] = None,
        show_progress: bool = True,
        desc: str = "COMET",
    ) -> List[float]:
        """
        Батчевый расчёт COMET-скоров.
        Если cache задан, то:
        кеш в процессе используется ВСЕГДА
        reuse_old_cache управляет тем, берём ли старые значения
            из файла/памяти или пересчитываем.
        """
        assert len(src_list) == len(ref_list) == len(cand_list)
        n = len(src_list)
        if batch_size is None:
            batch_size = self.default_batch_size

        all_scores: List[float] = []
        pbar = tqdm(total=n, desc=desc) if show_progress else None

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)

            # готовим данные для батча
            batch_data = []
            batch_hashes = []
            for src, ref, cand in zip(
                src_list[start:end],
                ref_list[start:end],
                cand_list[start:end],
            ):
                src_text = strip_tags_for_translation(src) if src is not None else ""
                ref_text = strip_tags_for_translation(ref)
                cand_text = strip_tags_for_translation(cand)

                batch_data.append({"src": src_text, "mt": cand_text, "ref": ref_text})
                batch_hashes.append(triple_hash(src_text, ref_text, cand_text))

            # распределяем: что уже есть в кеше, что надо считать
            scores_batch: List[Optional[float]] = [None] * len(batch_data)
            to_predict_data = []
            to_predict_indices = []

            for i, (h, item) in enumerate(zip(batch_hashes, batch_data)):
                cached = None
                if self.cache is not None and self.reuse_old_cache and self.cache.has(h):
                    cached = self.cache.get(h)

                if cached is not None:
                    scores_batch[i] = cached
                else:
                    to_predict_indices.append(i)
                    to_predict_data.append(item)

            # считаем только то, чего нет в кеше
            if to_predict_data:
                out = self.model.predict(
                    to_predict_data,
                    batch_size=len(to_predict_data),
                    gpus=1,
                    progress_bar=False,
                )
                predicted_scores = [float(s) for s in out["scores"]]

                for idx, s in zip(to_predict_indices, predicted_scores):
                    scores_batch[idx] = s
                    if self.cache is not None:
                        self.cache.add(batch_hashes[idx], s)

            all_scores.extend(float(s) for s in scores_batch)

            if pbar is not None:
                pbar.update(len(batch_data))

            if self.cache is not None:
                self.cache.flush()

        if pbar is not None:
            pbar.close()

        return all_scores


def _compute_tag_metrics_batch(
    ref_list: List[str],
    cand_list: List[str],
    show_progress: bool = True,
    desc: str = "Tags/compilability",
) -> Dict[str, List[float]]:
    """
    Рассчитывает батчем метрику по тегам
    """
    assert len(ref_list) == len(cand_list)
    n = len(ref_list)

    tag_content_scores: List[float] = []
    tag_struct_scores: List[float] = []
    tag_scores: List[float] = []
    comp_scores: List[float] = []

    pbar = tqdm(total=n, desc=desc) if show_progress else None

    for ref, cand in zip(ref_list, cand_list):
        ref_tags = summarize_tags(ref)
        cand_tags = summarize_tags(cand)

        tag_content = weighted_tag_content_score(ref_tags, cand_tags)
        tag_struct = tag_structure_score(ref_tags, cand_tags)
        tag_score = 0.6 * tag_content + 0.4 * tag_struct
        comp_score = compilability_score(cand_tags)

        tag_content_scores.append(tag_content)
        tag_struct_scores.append(tag_struct)
        tag_scores.append(tag_score)
        comp_scores.append(comp_score)

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    return {
        "tag_content_score": tag_content_scores,
        "tag_structure_score": tag_struct_scores,
        "tag_score": tag_scores,
        "compilability": comp_scores,
    }


@dataclass
class ScoreSetResult:
    """
    Просто контейнер для результатов.
    Все листы одной длины.
    """
    text_score: List[float]
    tag_content_score: List[float]
    tag_structure_score: List[float]
    tag_score: List[float]
    compilability: List[float]
    final_score: List[float]


class ScoreSet:
    """
    Принимает списки src/ref/cand и считает все метрики разом по батчам.
    """

    def __init__(
        self,
        refs: List[str],
        cands: List[str],
        srcs: Optional[List[Optional[str]]] = None,
        comet_scorer: Optional[ModelCometScorer] = None,
        precomputed_text_scores: Optional[List[Optional[float]]] = None,
        comet_batch_size: int = 128,
        show_progress: bool = True,
        parallel: bool = True,
        comet_cache_file: Optional[str] = None,
        reuse_old_comet_cache: bool = True,
    ):
        assert len(refs) == len(cands)
        n = len(refs)

        if srcs is None:
            srcs = [None] * n
        else:
            assert len(srcs) == n

        if precomputed_text_scores is not None:
            assert len(precomputed_text_scores) == n

        self.refs = refs
        self.cands = cands
        self.srcs = srcs


        if comet_scorer is not None:
            self.comet_scorer = comet_scorer
        else:
            self.comet_scorer = None

        self.precomputed_text_scores = precomputed_text_scores
        self.comet_batch_size = comet_batch_size
        self.show_progress = show_progress
        self.parallel = parallel

        self.comet_cache: Optional[CometFileCache] = None
        if comet_cache_file is not None:
            self.comet_cache = CometFileCache(
                path=comet_cache_file,
                use_existing=reuse_old_comet_cache,
                flush_every=1,
            )
            if self.comet_scorer is not None:
                self.comet_scorer.cache = self.comet_cache
                self.comet_scorer.reuse_old_cache = reuse_old_comet_cache

    def compute(self) -> ScoreSetResult:
        """
        Основной метод: считает все метрики и возвращает ScoreSetResult.
        Счёт идёт по батчам, COMET и тэги по батчу можно считать параллельно.
        """
        n = len(self.refs)

        text_scores: List[float] = []
        tag_content_scores: List[float] = []
        tag_structure_scores: List[float] = []
        tag_scores: List[float] = []
        comp_scores: List[float] = []
        final_scores: List[float] = []

        pbar = tqdm(total=n, desc="Scoring (COMET + tags)") if self.show_progress else None

        executor: Optional[ThreadPoolExecutor] = None
        if self.parallel:
            executor = ThreadPoolExecutor(max_workers=2)

        try:
            for start in range(0, n, self.comet_batch_size):
                end = min(start + self.comet_batch_size, n)

                refs_batch = self.refs[start:end]
                cands_batch = self.cands[start:end]
                srcs_batch = self.srcs[start:end]

                comet_scores_batch: Optional[List[float]] = None
                if self.precomputed_text_scores is not None:
                    comet_scores_batch = [
                        float(s) if s is not None else 0.0
                        for s in self.precomputed_text_scores[start:end]
                    ]
                elif self.comet_scorer is not None:
                    def run_comet():
                        return self.comet_scorer.score_batch(
                            src_list=srcs_batch,
                            ref_list=refs_batch,
                            cand_list=cands_batch,
                            batch_size=len(refs_batch),
                            show_progress=False,
                            desc="COMET(batch)",
                        )
                else:
                    def run_comet():
                        return [0.0] * len(refs_batch)

                def run_tags():
                    return _compute_tag_metrics_batch(
                        ref_list=refs_batch,
                        cand_list=cands_batch,
                        show_progress=False,
                        desc="Tags/compilability(batch)",
                    )

                if executor is not None and comet_scores_batch is None:
                    comet_future = executor.submit(run_comet)
                    tags_future = executor.submit(run_tags)

                    comet_scores_batch = comet_future.result()
                    tag_metrics_batch = tags_future.result()
                else:
                    if comet_scores_batch is None:
                        comet_scores_batch = run_comet()
                    tag_metrics_batch = run_tags()

                # Собираем финальные метрики по батчу
                tc_batch = tag_metrics_batch["tag_content_score"]
                ts_batch = tag_metrics_batch["tag_structure_score"]
                t_batch = tag_metrics_batch["tag_score"]
                comp_batch = tag_metrics_batch["compilability"]

                for i in range(len(refs_batch)):
                    text_score = float(comet_scores_batch[i])
                    tag_content = tc_batch[i]
                    tag_struct = ts_batch[i]
                    tag_score = t_batch[i]
                    comp_score = comp_batch[i]

                    final = comp_score * tag_score * text_penalty(text_score)

                    text_scores.append(text_score)
                    tag_content_scores.append(tag_content)
                    tag_structure_scores.append(tag_struct)
                    tag_scores.append(tag_score)
                    comp_scores.append(comp_score)
                    final_scores.append(final)

                if pbar is not None:
                    pbar.update(len(refs_batch))

        finally:
            if pbar is not None:
                pbar.close()
            if executor is not None:
                executor.shutdown(wait=True)

        return ScoreSetResult(
            text_score=text_scores,
            tag_content_score=tag_content_scores,
            tag_structure_score=tag_structure_scores,
            tag_score=tag_scores,
            compilability=comp_scores,
            final_score=final_scores,
        )
